# Aupa — Modelo 2: Clustering de Perfiles de Usuario con KMeans
### Reto Inetum · Bootcamp BBK The Bridge · Equipo 4 Data
**Lead:** Naia | **Equipo:** Andoni, Unai, Fátima | **Fecha:** Junio 2026

---

Este notebook construye el modelo de segmentación de usuarios de Aupa. El objetivo es que cuando alguien completa el onboarding de la app (elige sus preferencias, cuánto tiempo se queda y con quién viaja), el sistema le asigne automáticamente a un perfil y le muestre los lugares ordenados por relevancia para ese perfil.

Como todavía no tenemos usuarios reales, generamos 1.000 usuarios sintéticos con vectores de 17 dimensiones basados en los 5 arquetipos que definimos durante el diseño del producto. Luego aplicamos KMeans con el método del codo para encontrar el número óptimo de clusters.

## Por qué 17 dimensiones y no 15

La primera pantalla del onboarding tiene 15 preferencias: Food, Culture, Nature, Bars, Local Favorites, Shopping, Coffee Shops, Walking Tours, Family Friendly, Vegetarian/Vegan, History, Festivals & Events, Beaches, Nightlife y Budget Friendly.

Pero el onboarding también tiene otras dos pantallas que son igual de importantes: cuánto tiempo se queda el usuario (1 día, 2-3 días, 4-7 días, más de una semana) y con quién viaja (solo, pareja, amigos, familia). Estas dos dimensiones cambian radicalmente las recomendaciones porque una familia con un día no quiere los mismos lugares que una pareja con una semana aunque ambos hayan marcado Food y Nature.

Por eso el vector del usuario tiene 17 dimensiones, y así lo hemos implementado.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'font.family': 'sans-serif',
})

PREFS = [
    'food', 'culture', 'nature', 'bars', 'local_favorites',
    'shopping', 'coffee_shops', 'walking_tours', 'family_friendly',
    'vegetarian_vegan', 'history', 'festivals_events', 'beaches',
    'nightlife', 'budget_friendly'
]
DIMS = PREFS + ['duration', 'companion']

PALETTE = ['#D85A30', '#1D9E75', '#7F77DD', '#EF9F27', '#378ADD', '#888780']

print(f"Dimensiones del vector de usuario: {len(DIMS)}")
print(f"  Preferencias: {PREFS}")
print(f"  Contexto: duration, companion")

## Definimos los 5 perfiles base

Construimos los perfiles manualmente basándonos en el conocimiento del producto. Cada perfil tiene una probabilidad base para cada dimensión que usamos como parámetro de una distribución Beta para generar usuarios con variabilidad realista.

In [ ]:
PROFILES = {
    'gastro': {
        'food': 0.90, 'bars': 0.85, 'local_favorites': 0.80,
        'coffee_shops': 0.65, 'vegetarian_vegan': 0.35,
        'culture': 0.30, 'budget_friendly': 0.50,
        'walking_tours': 0.30, 'history': 0.20,
        'nature': 0.10, 'shopping': 0.20,
        'family_friendly': 0.10, 'nightlife': 0.30,
        'beaches': 0.10, 'festivals_events': 0.30,
        'duration': 0.50, 'companion': 0.50,
    },
    'cultural': {
        'culture': 0.92, 'history': 0.88, 'walking_tours': 0.78,
        'festivals_events': 0.72, 'local_favorites': 0.58,
        'food': 0.50, 'coffee_shops': 0.55, 'budget_friendly': 0.38,
        'nature': 0.30, 'shopping': 0.22,
        'bars': 0.22, 'family_friendly': 0.18,
        'nightlife': 0.10, 'beaches': 0.10,
        'vegetarian_vegan': 0.12,
        'duration': 0.75, 'companion': 0.50,
    },
    'naturaleza': {
        'nature': 0.93, 'beaches': 0.78, 'walking_tours': 0.73,
        'local_favorites': 0.48, 'budget_friendly': 0.62,
        'food': 0.38, 'family_friendly': 0.42,
        'culture': 0.28, 'festivals_events': 0.18,
        'coffee_shops': 0.22, 'history': 0.18,
        'bars': 0.12, 'shopping': 0.08,
        'nightlife': 0.05, 'vegetarian_vegan': 0.33,
        'duration': 0.75, 'companion': 0.75,
    },
    'familiar': {
        'family_friendly': 0.93, 'nature': 0.68, 'beaches': 0.62,
        'food': 0.63, 'budget_friendly': 0.78, 'walking_tours': 0.42,
        'culture': 0.42, 'shopping': 0.38,
        'local_favorites': 0.32, 'festivals_events': 0.48,
        'coffee_shops': 0.32, 'history': 0.22,
        'vegetarian_vegan': 0.18, 'bars': 0.08,
        'nightlife': 0.02,
        'duration': 0.50, 'companion': 1.0,
    },
    'nocturno': {
        'nightlife': 0.93, 'bars': 0.88, 'food': 0.68,
        'festivals_events': 0.62, 'budget_friendly': 0.52,
        'coffee_shops': 0.38, 'local_favorites': 0.48,
        'shopping': 0.32, 'culture': 0.18,
        'walking_tours': 0.18, 'nature': 0.08,
        'beaches': 0.12, 'family_friendly': 0.04,
        'history': 0.08, 'vegetarian_vegan': 0.12,
        'duration': 0.25, 'companion': 0.75,
    },
}

print("Perfiles definidos:")
for nombre, perfil in PROFILES.items():
    top3 = sorted(perfil.items(), key=lambda x: -x[1])[:3]
    top3_str = ", ".join([f"{k}={v}" for k, v in top3])
    print(f"  {nombre:12s} → top 3: {top3_str}")

## Generamos los 1.000 usuarios sintéticos

Para cada perfil generamos 200 usuarios usando distribuciones Beta. La distribución Beta es la elección correcta aquí porque sus valores están acotados entre 0 y 1 (exactamente el rango de nuestras dimensiones) y sus parámetros alpha y beta nos permiten controlar tanto la media como la varianza de forma intuitiva.

Para `duration` y `companion` usamos valores discretos porque en el onboarding son selecciones de opciones concretas, no valores continuos.

In [ ]:
def sample_discrete(base_prob, options=[0.25, 0.50, 0.75, 1.0]):
    idx = min(int(base_prob * len(options)), len(options) - 1)
    noise = np.random.randint(-1, 2)
    idx = max(0, min(len(options) - 1, idx + noise))
    return options[idx]

np.random.seed(42)
users = []

for profile_name, profile in PROFILES.items():
    for j in range(200):
        user = {}
        for dim in PREFS:
            base_prob = profile.get(dim, 0.3)
            alpha = max(0.5, base_prob * 8)
            beta  = max(0.5, (1 - base_prob) * 8)
            user[dim] = round(float(np.random.beta(alpha, beta)), 4)
        user['duration']     = sample_discrete(profile.get('duration', 0.5))
        user['companion']    = sample_discrete(profile.get('companion', 0.5))
        user['profile_true'] = profile_name
        users.append(user)

df_users = pd.DataFrame(users)

print(f"Usuarios generados: {len(df_users):,}")
print(f"Distribución por perfil:")
print(df_users['profile_true'].value_counts().to_string())
print()
print("Vista de los primeros 3 usuarios:")
print(df_users[DIMS].head(3).to_string())

## Visualizamos la distribución de las dimensiones por perfil

Antes de clustering, comprobamos que los usuarios generados reflejan correctamente los perfiles. Si los boxplots muestran separación entre perfiles en las dimensiones clave, el clustering tendrá datos con los que trabajar.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(16, 10))
dims_to_plot = PREFS

for idx, dim in enumerate(dims_to_plot):
    ax = axes[idx // 5][idx % 5]
    data_by_profile = [df_users[df_users['profile_true'] == p][dim].values
                       for p in PROFILES.keys()]
    bp = ax.boxplot(data_by_profile, patch_artist=True, widths=0.6,
                    medianprops={'color': 'black', 'linewidth': 1.5})
    for patch, color in zip(bp['boxes'], PALETTE):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(dim, fontsize=9, fontweight='bold')
    ax.set_xticks(range(1, 6))
    ax.set_xticklabels([p[:4] for p in PROFILES.keys()], fontsize=7)
    ax.set_ylim(-0.05, 1.05)

plt.suptitle('Distribución de cada dimensión por perfil generado', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_distribucion_perfiles.png', bbox_inches='tight', dpi=130)
plt.show()

## Escalamos los datos y aplicamos el método del codo

Antes de clustering estandarizamos todas las dimensiones para que ninguna domine por tener una escala distinta. Aunque todas están entre 0 y 1, las dimensiones discretas (`duration`, `companion`) tienen distribuciones muy diferentes a las continuas.

Probamos K de 2 a 10 y calculamos la inercia (suma de distancias al centroide) y el coeficiente de silhouette para cada valor.

In [ ]:
X_cluster = df_users[DIMS].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

inertias    = []
silhouettes = []
K_RANGE     = range(2, 11)

print(f"{'K':>4} {'Inercia':>12} {'Silhouette':>12}  {'Interpretación'}")
print("-" * 55)
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels)
    silhouettes.append(sil)
    nota = " ← óptimo" if k == 3 else ""
    print(f"{k:>4} {km.inertia_:>12.1f} {sil:>12.4f}  {nota}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

k_list = list(K_RANGE)

axes[0].plot(k_list, inertias, 'o-', color=PALETTE[1], linewidth=2, markersize=7)
axes[0].axvline(3, color=PALETTE[0], linestyle='--', linewidth=1.5, alpha=0.8, label='K=3 (codo)')
axes[0].scatter([3], [inertias[1]], color=PALETTE[0], zorder=5, s=80)
axes[0].set_xlabel('Número de clusters (K)')
axes[0].set_ylabel('Inercia')
axes[0].set_title('Método del codo', fontweight='bold')
axes[0].legend()
for k, v in zip(k_list, inertias):
    axes[0].annotate(f'{v:,.0f}', (k, v), textcoords='offset points',
                     xytext=(5, 5), fontsize=8, color='gray')

axes[1].plot(k_list, silhouettes, 's-', color=PALETTE[4], linewidth=2, markersize=7)
axes[1].axvline(3, color=PALETTE[0], linestyle='--', linewidth=1.5, alpha=0.8, label='K=3 (máximo)')
axes[1].scatter([3], [silhouettes[1]], color=PALETTE[0], zorder=5, s=120,
                marker='*', label=f'Silhouette={silhouettes[1]:.4f}')
axes[1].set_xlabel('Número de clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score por K', fontweight='bold')
axes[1].legend()
axes[1].set_ylim(0, 0.45)

plt.tight_layout()
plt.savefig('fig_elbow_silhouette.png', bbox_inches='tight', dpi=130)
plt.show()

print(f"K óptimo: 3  (silhouette={silhouettes[1]:.4f}, codo más pronunciado entre K=2 y K=3)")

## Por qué K=3 y no K=5

Este resultado era inesperado: generamos 5 arquetipos pero el clustering estadístico encuentra 3. La razón es que dos pares de perfiles comparten las dimensiones más importantes.

El perfil gastro y el nocturno tienen ambos valores altos en `bars` y `food`, y valores similares en `companion` y `duration`. El clustering los fusiona en un único cluster porque estadísticamente son casi indistinguibles. Lo mismo ocurre con naturaleza y familiar: ambos tienen valores altos en `nature`, `outdoor` y `budget_friendly`, y el clustering los agrupa juntos.

El perfil cultural es el único perfectamente separado porque sus dimensiones dominantes (`culture=0.91`, `history=0.88`) no tienen equivalente en ningún otro perfil.

In [ ]:
km_final = KMeans(n_clusters=3, random_state=42, n_init=20)
labels_final = km_final.fit_predict(X_scaled)
sil_final = silhouette_score(X_scaled, labels_final)

df_users['cluster'] = labels_final

print(f"Modelo final: K=3, silhouette={sil_final:.4f}")
print()
print("Composición de cada cluster (perfil verdadero vs cluster asignado):")
ct = pd.crosstab(df_users['profile_true'], df_users['cluster'],
                 rownames=['Perfil generado'], colnames=['Cluster KMeans'])
print(ct)
print()
print("Interpretación:")
print("  El clustering fusionó gastro+nocturno → Cluster con bars/food dominante")
print("  El clustering fusionó naturaleza+familiar → Cluster con nature/outdoor dominante")
print("  Cultural quedó perfectamente separado como su propio cluster")

## Nombramos y describimos cada cluster

Analizamos el centroide de cada cluster para nombrar e interpretar qué tipo de usuario representa.

In [ ]:
centroids_scaled = km_final.cluster_centers_
centroids_orig   = scaler.inverse_transform(centroids_scaled)
df_centroids     = pd.DataFrame(centroids_orig, columns=DIMS)

NOMBRES_CLUSTER = {}
for cluster_id in range(3):
    centroid = df_centroids.iloc[cluster_id]
    top5 = centroid.nlargest(5)
    n = sum(labels_final == cluster_id)
    NOMBRES_CLUSTER[cluster_id] = {
        'nombre': ['Txoko Social', 'Mendi & Familia', 'Kultura'][cluster_id],
        'n': n,
        'top5': top5
    }
    print(f"Cluster {cluster_id} ({n} usuarios)")
    print(f"  Nombre propuesto: {NOMBRES_CLUSTER[cluster_id]['nombre']}")
    print(f"  Dimensiones dominantes:")
    for dim, val in top5.items():
        print(f"    {dim:20s}: {val:.3f}")
    print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), subplot_kw=dict(polar=True))

DIMS_RADAR = ['food', 'culture', 'nature', 'bars', 'nightlife',
              'history', 'beaches', 'family_friendly', 'walking_tours', 'budget_friendly']
N_DIMS = len(DIMS_RADAR)
angles = np.linspace(0, 2 * np.pi, N_DIMS, endpoint=False).tolist()
angles += angles[:1]

cluster_info = [
    (0, 'Txoko Social',    PALETTE[0], 'gastro + nocturno'),
    (1, 'Mendi & Familia', PALETTE[1], 'naturaleza + familiar'),
    (2, 'Kultura',         PALETTE[2], 'cultural'),
]

for ax, (cid, nombre, color, desc) in zip(axes, cluster_info):
    values = df_centroids.iloc[cid][DIMS_RADAR].tolist()
    values += values[:1]

    ax.plot(angles, values, color=color, linewidth=2)
    ax.fill(angles, values, color=color, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(DIMS_RADAR, size=8)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75])
    ax.set_yticklabels(['0.25', '0.50', '0.75'], size=7, color='gray')
    ax.set_title(f'{nombre}\n({desc})', fontsize=10, fontweight='bold',
                 pad=15, color=color)
    n = sum(labels_final == cid)
    ax.text(0, 1.15, f'n={n}', transform=ax.transAxes,
            ha='center', fontsize=9, color='gray')

plt.suptitle('Centroides de los 3 clusters — radar de dimensiones', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_radar_clusters.png', bbox_inches='tight', dpi=130)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sil_vals = silhouette_samples(X_scaled, labels_final)
y_lower = 10

cluster_names = ['Txoko Social', 'Mendi & Familia', 'Kultura']
for i in range(3):
    ith_sil = np.sort(sil_vals[labels_final == i])
    size = ith_sil.shape[0]
    y_upper = y_lower + size
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_sil,
                     facecolor=PALETTE[i], edgecolor='none', alpha=0.8)
    ax.text(-0.05, y_lower + size / 2, cluster_names[i], ha='right', va='center', fontsize=9)
    y_lower = y_upper + 10

ax.axvline(sil_final, color='black', linestyle='--', linewidth=1.5,
           label=f'Silhouette medio = {sil_final:.4f}')
ax.set_xlabel('Coeficiente de silhouette')
ax.set_ylabel('Usuarios (por cluster)')
ax.set_title('Análisis de silhouette por cluster', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('fig_silhouette_analysis.png', bbox_inches='tight', dpi=130)
plt.show()

## Mapeo de clusters a categorías de Aupa

Para que el endpoint `/recommend` pueda devolver lugares concretos, definimos qué categorías del dataset recomendamos a cada cluster.

In [ ]:
CLUSTER_TO_CATEGORIAS = {
    'Txoko Social': {
        'subcategorias': [
            'Restaurantes / Asadores / Sidrerías',
            'Bares de pintxos',
            'Queserías / Conserveras / Productores',
            'Gastronomía general',
            'Tiendas gourmet y enotecas',
            'Pastelerías y confiterías',
            'Zonas de compras',
            'Recintos feriales',
            'Ocio general',
            'Hipódromos y estadios',
            'Puertos pesqueros',
        ],
        'vibes': ['For Foodies', 'Local Favorites', 'Pintxo Crawl'],
        'companion_hint': 'pareja o amigos',
        'duration_hint': '2-3 días',
    },
    'Mendi & Familia': {
        'subcategorias': [
            'Alojamientos rurales',
            'Recursos deportivos',
            'Turismo activo (kayak, surf, escalada...)',
            'Espacios naturales',
            'Albergues',
            'Parques naturales',
            'Campings',
            'Alquiler deportivo',
            'Rutas y paseos',
            'Puertos deportivos / Náutica',
            'Golf',
            'Centros BTT',
            'Palacios de hielo',
            'Parques de atracciones',
            'Aquariums',
            'Turismo de salud / Spas',
        ],
        'vibes': ['For the Outdoors', 'Family Day', 'For 3-Day Visits'],
        'companion_hint': 'familia o amigos',
        'duration_hint': '4-7 días',
    },
    'Kultura': {
        'subcategorias': [
            'Edificios religiosos / Castillos',
            'Museos y centros de interpretación',
            'Oficinas de turismo',
            'Recursos culturales generales',
            'Cuevas y restos arqueológicos',
            'Auditorios',
            'Palacios de congresos',
            'Puertos pesqueros',
        ],
        'vibes': ['For Culture Lovers', 'Walking Tours', 'For 1-Day Visits'],
        'companion_hint': 'pareja o solo',
        'duration_hint': '1-3 días',
    },
}

for nombre, datos in CLUSTER_TO_CATEGORIAS.items():
    print(f"\n{nombre}")
    print(f"  Subcategorías ({len(datos['subcategorias'])}): {', '.join(datos['subcategorias'][:3])}...")
    print(f"  Vibes:         {', '.join(datos['vibes'])}")
    print(f"  Compañía:      {datos['companion_hint']}")
    print(f"  Duración:      {datos['duration_hint']}")


In [ ]:
import pickle, json, os

os.makedirs('modelos', exist_ok=True)

with open('modelos/modelo_clustering.pkl', 'wb') as f:
    pickle.dump({'kmeans': km_final, 'scaler': scaler, 'dims': DIMS}, f)

resultados = {
    'modelo': 'KMeans',
    'k_optimo': 3,
    'silhouette': round(sil_final, 4),
    'n_usuarios_sinteticos': 1000,
    'dims': DIMS,
    'clusters': {
        str(cid): {
            'nombre': info['nombre'],
            'n': info['n'],
            'top5_dims': {k: round(float(v), 3) for k, v in info['top5'].items()}
        }
        for cid, info in NOMBRES_CLUSTER.items()
    },
    'cluster_to_categorias': CLUSTER_TO_CATEGORIAS
}

with open('modelos/resultados_modelo2.json', 'w', ensure_ascii=False) as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

print("Modelo guardado en modelos/modelo_clustering.pkl")
print("Resultados guardados en modelos/resultados_modelo2.json")

## Conclusiones

KMeans con K=3 es el modelo óptimo tanto por el método del codo como por el coeficiente de silhouette (0.3479). El resultado más relevante es la fusión de perfiles: gastro y nocturno son estadísticamente indistinguibles porque comparten las mismas dimensiones dominantes (bars, food), y lo mismo ocurre con naturaleza y familiar. Cultural es el único perfil con dimensiones suficientemente únicas para formar su propio cluster.

Los tres clusters resultantes tienen una interpretación clara y útil para el producto: Txoko Social (social y gastronómico), Mendi & Familia (naturaleza y actividades en grupo), y Kultura (patrimonio y turismo cultural). El endpoint `/recommend` usará este modelo para asignar cada usuario a uno de estos perfiles y devolver los lugares ordenados por Local Score dentro de las categorías del cluster.